In [1]:
"""
backward_model_nuts.ipynb — Tier 3: NUTS HMC with JIT Log-Prob
================================================================
Replaces Laplace approximation with NUTS (No-U-Turn Sampler) for
proper posterior uncertainty estimation on non-Gaussian distributions.

Two-stage FERRE-style:
  Stage 1 — 9D core physics (NUTS, full spectrum)
  Stage 2 — 14 × 1D abundances (NUTS, element-masked pixels)

CNN warm-start provides initialisation + Gaussian prior.

Run on Kaggle with GPU enabled.
"""

import os
import time
import json
import numpy as np
import h5py
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras import layers
from sklearn.experimental import enable_iterative_imputer   # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

tfb = tfp.bijectors

# ================================================================
# CONFIG (identical to backward_model_hmc.py)
# ================================================================

class Config:
    H5_PATH    = "/kaggle/input/datasets/aneeshshastri/aspcapstar-dr17-150kstars/apogee_dr17_parallel.h5"
    TFREC_DIR  = "/kaggle/working/tfrecords"
    STATS_PATH = "/kaggle/working/dataset_stats.npz"
    start = 140_000
    end   = 149_500
    TESTING_MODE  = True
    RANDOM_SEED   = 42
    OUTPUT_LENGTH = 8575
    BADPIX_CUTOFF = 1e-3
    SELECTED_LABELS = [
        'TEFF', 'LOGG', 'M_H', 'VMICRO', 'VMACRO', 'VSINI',
        'C_FE', 'N_FE', 'O_FE',
        'FE_H',
        'MG_FE', 'SI_FE', 'CA_FE', 'TI_FE', 'S_FE',
        'AL_FE', 'MN_FE', 'NI_FE', 'CR_FE', 'K_FE',
        'NA_FE', 'V_FE', 'CO_FE',
    ]
    ABUNDANCE_INDICES = [i for i, l in enumerate(SELECTED_LABELS) if '_FE' in l]
    FE_H_INDEX = SELECTED_LABELS.index('FE_H')
    N_LABELS   = len(SELECTED_LABELS) + 4
    ERRORS     = [f'{l}_ERR' for l in SELECTED_LABELS]

config = Config()
np.random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)

# ================================================================
# CUSTOM LAYERS (needed for model loading — identical to HMC file)
# ================================================================

@register_keras_serializable()
def chi2_estimate(y_true,y_pred):
    y_true=tf.cast(y_true,tf.float32)
    y_pred=tf.cast(y_pred,tf.float32)
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux = y_true[:, :, 0:1]
    ivar = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)    
    safe_flux = tf.where(valid_mask == 1.0, real_flux, y_pred)
    weight=ivar
    weight=tf.where(ivar>0,weight,0.0)
    wmse_term = tf.square(safe_flux - y_pred) * weight * valid_mask
    n_valid_pixels = tf.reduce_sum(valid_mask, 1)
    n_params = len(config.SELECTED_LABELS)
    dof = n_valid_pixels - n_params
    return tf.reduce_sum(wmse_term,1)/dof


@register_keras_serializable()
def spl_loss(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux = y_true[:, :, 0:1]
    ivar = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)    
    safe_flux = tf.where(valid_mask == 1.0, real_flux, y_pred)
    weight=1e-5/(1/ivar+1e-5)
    weight=tf.where(ivar>0,weight,0.0)
    wmse_term = tf.square(safe_flux - y_pred) * weight * valid_mask
    loss = tf.where(tf.math.is_finite(wmse_term), wmse_term, tf.zeros_like(wmse_term))
    return loss

@register_keras_serializable()
class ColumnSelector(layers.Layer):
    def __init__(self, indices, **kwargs):
        super().__init__(**kwargs)
        self.indices = list(indices)
    def call(self, inputs):
        return tf.gather(inputs, self.indices, axis=1)
    def get_config(self):
        c = super().get_config(); c['indices'] = self.indices; return c

@register_keras_serializable()
class BeerLambertLaw(layers.Layer):
    def __init__(self, **kwargs): super().__init__(**kwargs)
    def call(self, k, tau): return k * tf.math.exp(-tau)

@register_keras_serializable()
class GetAbundances(layers.Layer):
    def __init__(self, col_id, **kwargs):
        super().__init__(**kwargs)
        self.index = col_id
    def call(self, inputs):
        return inputs[:, self.index:self.index+1]
    def get_config(self):
        c = super().get_config(); c['col_id'] = self.index; return c

@register_keras_serializable()
class SparseProjector(tf.keras.layers.Layer):
    def __init__(self, active_indices, weights, total_pixels, label_name="unknown", **kwargs):
        kwargs['name'] = f"Sparse_Projector_{label_name}"
        super().__init__(**kwargs)
        self.total_pixels = total_pixels
        self.label_name = label_name
        self.active_indices = tf.constant(active_indices, dtype=tf.int32)
        self.weights_tensor = tf.constant(weights, dtype=tf.float32)
    def call(self, local_tau):
        local_tau_32 = tf.cast(local_tau, tf.float32)
        weighted = local_tau_32 * self.weights_tensor[None, :]
        batch_size = tf.shape(local_tau)[0]
        n_active = tf.shape(self.active_indices)[0]
        batch_ids = tf.repeat(tf.range(batch_size), n_active)
        pixel_ids = tf.tile(self.active_indices, tf.expand_dims(batch_size, axis=0))
        scatter_idx = tf.stack([batch_ids, pixel_ids], axis=-1)
        return tf.scatter_nd(scatter_idx, tf.reshape(weighted, [-1]),
                             shape=tf.stack([batch_size, tf.constant(self.total_pixels, dtype=tf.int32)]))
    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.total_pixels)
    def get_config(self):
        c = super().get_config()
        c.update({'active_indices': self.active_indices.numpy().tolist(),
                  'weights': self.weights_tensor.numpy().tolist(),
                  'total_pixels': self.total_pixels, 'label_name': self.label_name})
        return c
    @classmethod
    def from_config(cls, cfg):
        cfg['active_indices'] = np.array(cfg['active_indices'], dtype=np.int32)
        cfg['weights'] = np.array(cfg['weights'], dtype=np.float32)
        cfg.pop('name', None)
        return cls(**cfg)

N_LABELS_RAW = 23
N_PIXELS     = 8575

@register_keras_serializable()
class HeteroscedasticCNNPredictor(tf.keras.Model):
    def __init__(self, n_labels=N_LABELS_RAW, **kwargs):
        super().__init__(**kwargs)
        self.n_labels = n_labels
        self.reshape_in = layers.Reshape((N_PIXELS, 1))
        self.conv1 = layers.Conv1D(32, 16, strides=4, activation='relu', padding='same')
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv1D(64, 8, strides=4, activation='relu', padding='same')
        self.bn2 = layers.BatchNormalization()
        self.conv3 = layers.Conv1D(128, 4, strides=2, activation='relu', padding='same')
        self.bn3 = layers.BatchNormalization()
        self.conv4 = layers.Conv1D(256, 4, strides=2, activation='relu', padding='same')
        self.bn4 = layers.BatchNormalization()
        self.gap = layers.GlobalAveragePooling1D()
        self.dense1 = layers.Dense(512, activation='relu')
        self.drop1 = layers.Dropout(0.3)
        self.dense2 = layers.Dense(256, activation='relu')
        self.drop2 = layers.Dropout(0.2)
        self.mean_head = layers.Dense(n_labels, name='label_mean')
        self.log_var_head = layers.Dense(n_labels, name='label_log_var')
    def call(self, x, training=False):
        h = self.reshape_in(x)
        h = self.bn1(self.conv1(h), training=training)
        h = self.bn2(self.conv2(h), training=training)
        h = self.bn3(self.conv3(h), training=training)
        h = self.bn4(self.conv4(h), training=training)
        h = self.gap(h)
        h = self.drop1(self.dense1(h), training=training)
        h = self.drop2(self.dense2(h), training=training)
        return self.mean_head(h), self.log_var_head(h)
    def get_config(self):
        c = super().get_config(); c['n_labels'] = self.n_labels; return c

@register_keras_serializable()
class TrainableCNNPredictor(HeteroscedasticCNNPredictor):
    pass

# ================================================================
# DATA LOADING (same as HMC file)
# ================================================================

def get_clean_imputed_data(h5_path, selected_labels, limit=None):
    with h5py.File(h5_path, 'r') as f:
        get_col = lambda k: f['metadata'][k]
        keys = f['metadata'].dtype.names
        raw_values = np.stack([get_col(p) for p in selected_labels], axis=1)
        bad_mask = np.zeros_like(raw_values, dtype=bool)
        for i, label in enumerate(selected_labels):
            flag_name = f"{label}_FLAG"
            if flag_name in keys:
                flg = get_col(flag_name)
                if flg.dtype.names: flg = flg[flg.dtype.names[0]]
                if flg.dtype.kind == 'V': flg = flg.view('<i4')
                bad_mask[:, i] = (flg.astype(int) != 0)
            elif label in ['TEFF', 'LOGG', 'VMICRO', 'VMACRO', 'VSINI']:
                bad_mask[:, i] = (raw_values[:, i] < -5000)
    if limit:
        raw_values = raw_values[:limit]; bad_mask = bad_mask[:limit]
    vals = raw_values.copy(); vals[bad_mask] = np.nan
    imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10, initial_strategy='median')
    clean = imputer.fit_transform(vals)
    fe_h = clean[:, config.FE_H_INDEX:config.FE_H_INDEX+1]
    for idx in config.ABUNDANCE_INDICES: clean[:, idx] += fe_h[:, 0]
    t = clean[:, selected_labels.index('TEFF')]
    clean = np.hstack([clean, (5040.0/(t+1e-6)).reshape(-1,1)])
    vm = clean[:, selected_labels.index('VMACRO')]
    vs = clean[:, selected_labels.index('VSINI')]
    clean = np.hstack([clean, np.sqrt(vm**2+vs**2).reshape(-1,1)])
    c = clean[:, selected_labels.index('C_FE')]
    o = clean[:, selected_labels.index('O_FE')]
    clean = np.hstack([clean, (c-o).reshape(-1,1)])
    g = clean[:, selected_labels.index('LOGG')]
    m = clean[:, selected_labels.index('M_H')]
    clean = np.hstack([clean, (0.5*g+0.5*m).reshape(-1,1)])
    return clean

def get_err(h5_path, selected_labels, limit=None):
    with h5py.File(h5_path, 'r') as f:
        get_col = lambda k: f['metadata'][k]
        keys = f['metadata'].dtype.names
        raw = np.stack([get_col(p) if p not in ['VMACRO_ERR','VMICRO_ERR','VSINI_ERR']
                        else np.zeros_like(get_col('TEFF_ERR')) for p in selected_labels], axis=1)
        bad = np.zeros_like(raw, dtype=bool)
        for i, l in enumerate(selected_labels):
            fn = f"{l}_FLAG"
            if fn in keys:
                flg = get_col(fn)
                if flg.dtype.names: flg = flg[flg.dtype.names[0]]
                if flg.dtype.kind == 'V': flg = flg.view('<i4')
                bad[:, i] = (flg.astype(int) != 0)
            elif l in ['TEFF','LOGG','VMICRO','VMACRO','VSINI']:
                bad[:, i] = (raw[:, i] < -5000)
        if limit: raw = raw[:limit]; bad = bad[:limit]
    v = raw.copy(); v[bad] = np.nan
    p95 = np.nanpercentile(v, 95, axis=0)
    return np.where(np.isnan(v), p95, v)

# ================================================================
# LOAD DATA + MODELS (paths from laplace notebook v15)
# ================================================================

print("Loading data...")
h5_path    = config.H5_PATH
stats_path = "/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/15/dataset_stats_120k.npz"
model_path = "/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/15/ThePayne.keras"
cnn_model_path = "/kaggle/input/models/aneeshshastri/backward-warmstart/tensorflow2/default/1/cnn_label_predictor_inner (1).keras"
cnn_stats_path = "/kaggle/input/models/aneeshshastri/backward-warmstart/tensorflow2/default/1/cnn_label_stats.npz"
apogee_mask_path = "/kaggle/input/datasets/aneeshshastri/element-masks/apogee_inference_mask.npy"

with h5py.File(h5_path, 'r') as f:
    real_data      = f['flux'][config.start:config.end]
    real_data_ivar = f['ivar'][config.start:config.end]

with np.load(stats_path) as data:
    MEAN_TENSOR = data['mean'].astype(np.float32)
    STD_TENSOR  = data['std'].astype(np.float32)

labels    = get_clean_imputed_data(h5_path, config.SELECTED_LABELS)
label_err = get_err(h5_path, config.ERRORS)[config.start:config.end]
labels    = ((labels - MEAN_TENSOR) / STD_TENSOR)[config.start:config.end]

model_raw = tf.keras.models.load_model(model_path)
cfg_m = model_raw.get_config()
cfg_str = json.dumps(cfg_m).replace('"float16"', '"float32"').replace('"mixed_float16"', '"float32"')
model_pure_fp32 = model_raw.__class__.from_config(json.loads(cfg_str))
model_pure_fp32.set_weights(model_raw.get_weights())
print("Forward model rebuilt in FP32.")

cnn_predictor = tf.keras.models.load_model(cnn_model_path)
with np.load(cnn_stats_path) as cs:
    CNN_LABEL_MEAN = cs['mean'].astype(np.float32)
    CNN_LABEL_STD  = cs['std'].astype(np.float32)

def cnn_predict_physical(flux_batch):
    fc = flux_batch.copy(); bad = ~np.isfinite(fc) | (fc <= 1e-3); fc[bad] = 1.0
    mu_n, rlv = cnn_predictor(tf.constant(fc, tf.float32), training=False)
    mu_p = mu_n.numpy() * CNN_LABEL_STD + CNN_LABEL_MEAN
    std_p = np.sqrt(tf.nn.softplus(rlv).numpy() + 1e-4) * CNN_LABEL_STD
    std_p = np.maximum(std_p, 0.01 * CNN_LABEL_STD)
    return mu_p.astype(np.float32), std_p.astype(np.float32)


2026-04-16 15:21:28.899987: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776352889.083860      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776352889.137813      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776352889.570227      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776352889.570275      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776352889.570278      23 computation_placer.cc:177] computation placer alr

Loading data...


I0000 00:00:1776353001.317049      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Forward model rebuilt in FP32.


In [2]:
# ================================================================
# CONSTANTS — NUTS HYPERPARAMETERS
# ================================================================

LABEL_NAMES  = config.SELECTED_LABELS
BATCH_SIZE_STARS = 10   # Run inference on 10 stars
TOTAL_STARS      = 10   # Only 10 stars for this run

CORE_INDICES  = [0,1,2,3,4,5,6,7,8]
N_CORE        = len(CORE_INDICES)
ABUND_INDICES = [9,10,11,12,13,14,15,16,17,18,19,20,21,22]
N_ABUND       = len(ABUND_INDICES)

# NUTS hyperparameters — generous budget for convergence
# NUTS hyperparameters — generous budget for convergence
NUM_CHAINS_CORE  = 4;   NUM_RESULTS_CORE = 200;  NUM_BURNIN_CORE = 200;  NUM_ADAPT_CORE = 150
NUM_CHAINS_ELEM  = 2;   NUM_RESULTS_ELEM = 100;  NUM_BURNIN_ELEM = 100;  NUM_ADAPT_ELEM = 80
MAX_TREE_DEPTH   = 10;   TARGET_ACCEPT    = 0.9
PRIOR_WEIGHT     = 1.0

RESULTS_DIR     = "/kaggle/working/nuts_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(RESULTS_DIR, "checkpoint.npz")
RESULTS_PATH    = os.path.join(RESULTS_DIR, "results.npz")

lower_bounds = [3000.,-0.5,-2.5,0.,0.,-20.,-2.5,-2.5,-2.5,-2.5,-2.0,-1.5,-1.5,-2.0,-2.0,-2.0,-2.0,-2.0,-2.5,-2.0,-2.0,-2.0,-2.0]
upper_bounds = [25000.,6.0,2.0,6.0,25.0,150.0,3.0,4.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,5.0,3.0,4.0]
bounds_low  = tf.constant(lower_bounds, tf.float32)
bounds_high = tf.constant(upper_bounds, tf.float32)

_raw_mask = np.load(apogee_mask_path)
ELEMENT_PIXEL_MASKS = {}
for idx in ABUND_INDICES:
    s = (_raw_mask[:, idx] > 0.01).astype(np.float32)
    if s.sum() < 10: s = np.ones(N_PIXELS, np.float32)
    ELEMENT_PIXEL_MASKS[idx] = tf.constant(s, tf.float32)

# ================================================================
# FEATURE ENGINEERING
# ================================================================

def get_27_features(labels_23):
    teff   = labels_23[:, config.SELECTED_LABELS.index('TEFF')]
    logg   = labels_23[:, config.SELECTED_LABELS.index('LOGG')]
    vmacro = labels_23[:, config.SELECTED_LABELS.index('VMACRO')]
    vsini  = labels_23[:, config.SELECTED_LABELS.index('VSINI')]
    c_fe   = labels_23[:, config.SELECTED_LABELS.index('C_FE')]
    o_fe   = labels_23[:, config.SELECTED_LABELS.index('O_FE')]
    m_h    = labels_23[:, config.SELECTED_LABELS.index('M_H')]
    eng = tf.stack([
        5040.0 / (teff + 1e-6),
        tf.sqrt(tf.square(vmacro) + tf.square(vsini)),
        c_fe - o_fe,
        0.5 * logg + 0.5 * m_h,
    ], axis=-1)
    return tf.concat([labels_23, eng], axis=-1)

# ================================================================
# TF.VARIABLE STATE FOR NUTS
# ================================================================

_cnn_mu_var  = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False)
_cnn_std_var = tf.Variable(tf.ones([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False)
_fixed_abund_var = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_ABUND], tf.float32), trainable=False)
_fixed_full_var  = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False)
_elem_col_var    = tf.Variable(0, dtype=tf.int32, trainable=False)
_elem_pixel_mask_var = tf.Variable(tf.ones([N_PIXELS], tf.float32), trainable=False)
_obs_flux_var = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_PIXELS], tf.float32), trainable=False)
_obs_ivar_var = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_PIXELS], tf.float32), trainable=False)

# ================================================================
# LOG PROBABILITY — JIT COMPILED
# ================================================================

@tf.function(reduce_retracing=True)
def _assemble_full_23(core_params, fixed_abund):
    parts = []
    for i in range(N_LABELS_RAW):
        if i in CORE_INDICES:
            parts.append(core_params[:, CORE_INDICES.index(i):CORE_INDICES.index(i)+1])
        else:
            parts.append(fixed_abund[:, ABUND_INDICES.index(i):ABUND_INDICES.index(i)+1])
    return tf.concat(parts, axis=1)

@tf.function(jit_compile=True)
def _forward_model_jit(labels_norm):
    """XLA-compiled forward model call."""
    model_flux = model_pure_fp32(labels_norm, training=False)
    if(len(model_flux.shape)>2):
        model_flux=model_flux[:,:,0]
    return tf.cast(model_flux, tf.float32)

@tf.function(reduce_retracing=True)
def core_log_prob_fn(theta_core, obs_flux, obs_ivar):
    """theta_core: (C, B, 9) physical -> (C, B)."""
    num_chains  = tf.shape(theta_core)[0]
    batch_stars = tf.shape(theta_core)[1]
    theta_BCL  = tf.transpose(theta_core, [1, 0, 2])
    theta_flat = tf.reshape(theta_BCL, [-1, N_CORE])
    abund_rep  = tf.repeat(_fixed_abund_var, num_chains, axis=0)
    full_23    = _assemble_full_23(theta_flat, abund_rep)
    labels_27   = get_27_features(full_23)
    labels_norm = (labels_27 - MEAN_TENSOR) / STD_TENSOR
    model_flux = _forward_model_jit(labels_norm)
    obs_flux_rep = tf.repeat(obs_flux, num_chains, axis=0)
    obs_ivar_rep = tf.repeat(obs_ivar, num_chains, axis=0)
    safe_flux = tf.where(tf.math.is_finite(obs_flux_rep), obs_flux_rep, 0.0)
    safe_ivar = tf.where(
        tf.math.is_finite(obs_ivar_rep) & tf.math.is_finite(obs_flux_rep),
        obs_ivar_rep, 0.0)
    safe_ivar = 1.0 / (1.0 / (safe_ivar + 1e-12) + 1e-5)
    fv = tf.cast((safe_flux > 1e-3) & (safe_flux < 1.3), tf.float32)
    iv = tf.cast(safe_ivar > 0.0, tf.float32)
    mask = fv * iv
    chi2 = tf.reduce_sum(tf.square(safe_flux - model_flux) * safe_ivar * mask, axis=1)
    log_lik = -0.5 * chi2
    cnn_mu_rep  = tf.repeat(tf.gather(_cnn_mu_var, CORE_INDICES, axis=1), num_chains, axis=0)
    cnn_std_rep = tf.repeat(tf.gather(_cnn_std_var, CORE_INDICES, axis=1), num_chains, axis=0)
    log_prior = -0.5 * tf.reduce_sum(tf.square((theta_flat - cnn_mu_rep) / cnn_std_rep), axis=1)
    log_post = log_lik + PRIOR_WEIGHT * log_prior
    return tf.transpose(tf.reshape(log_post, [batch_stars, num_chains]))

@tf.function(reduce_retracing=True)
def element_log_prob_fn(theta_1d, obs_flux, obs_ivar):
    """theta_1d: (C, B, 1) physical -> (C, B)."""
    num_chains  = tf.shape(theta_1d)[0]
    batch_stars = tf.shape(theta_1d)[1]
    elem_col    = _elem_col_var
    theta_BCL  = tf.transpose(theta_1d, [1, 0, 2])
    theta_flat = tf.reshape(theta_BCL, [-1, 1])
    full_rep = tf.repeat(_fixed_full_var, num_chains, axis=0)
    bc = tf.shape(full_rep)[0]
    indices = tf.stack([tf.range(bc), tf.fill([bc], elem_col)], axis=1)
    full_spliced = tf.tensor_scatter_nd_update(full_rep, indices, theta_flat[:, 0])
    labels_27   = get_27_features(full_spliced)
    labels_norm = (labels_27 - MEAN_TENSOR) / STD_TENSOR
    model_flux  = _forward_model_jit(labels_norm)
    obs_flux_rep = tf.repeat(obs_flux, num_chains, axis=0)
    obs_ivar_rep = tf.repeat(obs_ivar, num_chains, axis=0)
    safe_flux = tf.where(tf.math.is_finite(obs_flux_rep), obs_flux_rep, 0.0)
    safe_ivar = tf.where(
        tf.math.is_finite(obs_ivar_rep) & tf.math.is_finite(obs_flux_rep),
        obs_ivar_rep, 0.0)
    safe_ivar = 1.0 / (1.0 / (safe_ivar + 1e-12) + 1e-5)
    fv = tf.cast((safe_flux > 1e-3) & (safe_flux < 1.3), tf.float32)
    iv = tf.cast(safe_ivar > 0.0, tf.float32)
    mask = fv * iv * tf.reshape(_elem_pixel_mask_var, [1, N_PIXELS])
    chi2 = tf.reduce_sum(tf.square(safe_flux - model_flux) * safe_ivar * mask, axis=1)
    log_lik = -0.5 * chi2
    cnn_mu_rep  = tf.repeat(_cnn_mu_var[:, elem_col:elem_col+1], num_chains, axis=0)
    cnn_std_rep = tf.repeat(_cnn_std_var[:, elem_col:elem_col+1], num_chains, axis=0)
    log_prior = -0.5 * tf.reduce_sum(tf.square((theta_flat - cnn_mu_rep) / cnn_std_rep), axis=1)
    log_post = log_lik + PRIOR_WEIGHT * log_prior
    return tf.transpose(tf.reshape(log_post, [batch_stars, num_chains]))

# ================================================================
# BIJECTORS + INIT HELPERS
# ================================================================

_core_low_tf  = tf.gather(bounds_low, CORE_INDICES)
_core_high_tf = tf.gather(bounds_high, CORE_INDICES)
_core_bijector = tfb.Chain([tfb.Shift(_core_low_tf), tfb.Scale(_core_high_tf - _core_low_tf), tfb.Sigmoid()])
_core_inv_bijector = tfb.Invert(_core_bijector)

def make_init_state_core(batch_stars, num_chains, physical_init_23):
    margin = 1e-3
    core_init = tf.gather(tf.cast(physical_init_23, tf.float32), CORE_INDICES, axis=1)
    base = tf.tile(tf.expand_dims(core_init, 0), [num_chains, 1, 1])
    core_std = tf.gather(STD_TENSOR[:N_LABELS_RAW], CORE_INDICES)
    jitter = tf.random.normal(tf.shape(base)) * (0.05 * core_std)
    return tf.clip_by_value(base + jitter,
                            tf.gather(bounds_low + margin, CORE_INDICES),
                            tf.gather(bounds_high - margin, CORE_INDICES))

def make_init_state_elem(batch_stars, num_chains, phys_value_1d):
    base = tf.reshape(tf.cast(phys_value_1d, tf.float32), [1, -1, 1])
    base = tf.tile(base, [num_chains, 1, 1])
    return base + tf.random.normal(tf.shape(base)) * 0.02

# ================================================================
# NUTS SAMPLING
# ================================================================

_init_core_var = tf.Variable(
    tf.zeros([NUM_CHAINS_CORE, BATCH_SIZE_STARS, N_CORE], tf.float32), trainable=False)

#@tf.function(reduce_retracing=True)
def sample_core_compiled():
    """NUTS for Stage 1: 9D core physics."""
    def log_prob_closure(theta_unc):
        return core_log_prob_fn(
            _core_bijector.forward(theta_unc), _obs_flux_var, _obs_ivar_var)
    step_size = tf.fill([1, BATCH_SIZE_STARS, N_CORE], 0.05)
    nuts = tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=log_prob_closure,
        step_size=step_size,
        max_tree_depth=MAX_TREE_DEPTH,
    )
    adaptive = tfp.mcmc.DualAveragingStepSizeAdaptation(
        inner_kernel=nuts,
        num_adaptation_steps=NUM_ADAPT_CORE,
        target_accept_prob=TARGET_ACCEPT,
        step_size_setter_fn=lambda pkr, s: pkr._replace(step_size=s),
        step_size_getter_fn=lambda pkr: pkr.step_size,
        log_accept_prob_getter_fn=lambda pkr: pkr.log_accept_ratio,
    )
    init_unc = _core_inv_bijector.forward(_init_core_var)
    return tfp.mcmc.sample_chain(
        num_results=NUM_RESULTS_CORE,
        num_burnin_steps=NUM_BURNIN_CORE,
        current_state=init_unc,
        kernel=adaptive,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted,
    )

_elem_low_var  = tf.Variable(-3.0, dtype=tf.float32, trainable=False)
_elem_high_var = tf.Variable(3.0, dtype=tf.float32, trainable=False)
_init_elem_var = tf.Variable(
    tf.zeros([NUM_CHAINS_ELEM, BATCH_SIZE_STARS, 1], tf.float32), trainable=False)

#@tf.function(reduce_retracing=True)
def sample_element_compiled():
    """NUTS for Stage 2: 1D abundance."""
    lo = tf.reshape(_elem_low_var, [1]); hi = tf.reshape(_elem_high_var, [1])
    elem_bij = tfb.Chain([tfb.Shift(lo), tfb.Scale(hi - lo), tfb.Sigmoid()])
    elem_inv = tfb.Invert(elem_bij)
    def log_prob_closure(theta_unc):
        return element_log_prob_fn(
            elem_bij.forward(theta_unc), _obs_flux_var, _obs_ivar_var)
    step_size = tf.fill([1, BATCH_SIZE_STARS, 1], 0.1)
    nuts = tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=log_prob_closure,
        step_size=step_size,
        max_tree_depth=MAX_TREE_DEPTH,
    )
    adaptive = tfp.mcmc.DualAveragingStepSizeAdaptation(
        inner_kernel=nuts,
        num_adaptation_steps=NUM_ADAPT_ELEM,
        target_accept_prob=TARGET_ACCEPT,
        step_size_setter_fn=lambda pkr, s: pkr._replace(step_size=s),
        step_size_getter_fn=lambda pkr: pkr.step_size,
        log_accept_prob_getter_fn=lambda pkr: pkr.log_accept_ratio,
    )
    init_unc = elem_inv.forward(_init_elem_var)
    return tfp.mcmc.sample_chain(
        num_results=NUM_RESULTS_ELEM,
        num_burnin_steps=NUM_BURNIN_ELEM,
        current_state=init_unc,
        kernel=adaptive,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted,
    )

# ================================================================
# MCMC RUNNERS
# ================================================================

def run_core_mcmc(obs_flux, obs_ivar, physical_init_23, cnn_mean, cnn_std):
    _cnn_mu_var.assign(tf.cast(cnn_mean, tf.float32))
    _cnn_std_var.assign(tf.cast(cnn_std, tf.float32))
    _fixed_abund_var.assign(tf.gather(tf.cast(cnn_mean, tf.float32), ABUND_INDICES, axis=1))
    init_core = make_init_state_core(BATCH_SIZE_STARS, NUM_CHAINS_CORE, physical_init_23)
    _obs_flux_var.assign(tf.cast(obs_flux, tf.float32))
    _obs_ivar_var.assign(tf.cast(obs_ivar, tf.float32))
    _init_core_var.assign(init_core)

    samples_unc, is_accepted = sample_core_compiled()
    samples_phys = _core_bijector.forward(samples_unc)
    r_hat = tf.reduce_mean(
        tfp.mcmc.diagnostic.potential_scale_reduction(samples_phys, independent_chain_ndims=1), axis=0
    ).numpy()
    flat = tf.reshape(samples_phys, [NUM_RESULTS_CORE * NUM_CHAINS_CORE, -1, N_CORE])
    pcts = tfp.stats.percentile(flat, [16., 50., 84.], axis=0)
    return pcts[1].numpy(), (pcts[1]-pcts[0]).numpy(), (pcts[2]-pcts[1]).numpy(), \
           tf.reduce_mean(tf.cast(is_accepted, tf.float32)).numpy(), r_hat

def run_element_mcmc(obs_flux, obs_ivar, fixed_labels_23, elem_idx, cnn_mean, cnn_std):
    _fixed_full_var.assign(tf.cast(fixed_labels_23, tf.float32))
    _elem_col_var.assign(elem_idx)
    _elem_pixel_mask_var.assign(ELEMENT_PIXEL_MASKS[elem_idx])
    _elem_low_var.assign(bounds_low[elem_idx])
    _elem_high_var.assign(bounds_high[elem_idx])
    init_state = make_init_state_elem(BATCH_SIZE_STARS, NUM_CHAINS_ELEM, cnn_mean[:, elem_idx])
    _obs_flux_var.assign(tf.cast(obs_flux, tf.float32))
    _obs_ivar_var.assign(tf.cast(obs_ivar, tf.float32))
    _init_elem_var.assign(init_state)

    samples_unc, is_accepted = sample_element_compiled()
    lo_val = float(lower_bounds[elem_idx])
    hi_val = float(upper_bounds[elem_idx])
    bij = tfb.Chain([tfb.Shift(tf.constant([lo_val], tf.float32)), tfb.Scale(tf.constant([hi_val - lo_val], tf.float32)), tfb.Sigmoid()])
    samples_phys = bij.forward(samples_unc)
    r_hat = tf.reduce_mean(
        tfp.mcmc.diagnostic.potential_scale_reduction(samples_phys, independent_chain_ndims=1)).numpy()
    flat = tf.reshape(samples_phys, [NUM_RESULTS_ELEM * NUM_CHAINS_ELEM, -1, 1])
    pcts = tfp.stats.percentile(flat, [16., 50., 84.], axis=0)
    return pcts[1,:,0].numpy(), (pcts[1,:,0]-pcts[0,:,0]).numpy(), (pcts[2,:,0]-pcts[1,:,0]).numpy(), \
           tf.reduce_mean(tf.cast(is_accepted, tf.float32)).numpy(), r_hat

# ================================================================
# CHECKPOINT / RESULTS
# ================================================================

def load_checkpoint():
    fresh = {'global_indices':[],'true_labels':[],'inferred_labels':[],
             'lower_sigma':[],'upper_sigma':[],'aspcap_errors':[],
             'acceptance_rates':[],'r_hat':[],'wall_seconds':[]}
    if os.path.exists(CHECKPOINT_PATH):
        data = np.load(CHECKPOINT_PATH, allow_pickle=True)
        r = {k: list(data[k]) for k in fresh}
        print(f"  Resuming: {len(r['global_indices'])} done.")
        return r, len(r['global_indices'])
    return fresh, 0

def save_checkpoint(results):
    np.savez(CHECKPOINT_PATH, **{k: np.array(v) for k, v in results.items()})

def save_final_results(results):
    a = {k: np.array(v) for k, v in results.items()}
    a['label_names'] = np.array(LABEL_NAMES)
    np.savez(RESULTS_PATH, **a)
    print(f"\nFinal results saved to {RESULTS_PATH}")

# ================================================================
# TWO-STAGE INFERENCE PIPELINE (NUTS)
# ================================================================

def run_inference_pipeline(test_indices, true_labels_norm, aspcap_errors,
                           flux_array, ivar_array):
    n_stars = len(test_indices)
    true_phys = true_labels_norm[:, :N_LABELS_RAW] * STD_TENSOR[:N_LABELS_RAW] + MEAN_TENSOR[:N_LABELS_RAW]
    results, n_done = load_checkpoint()
    remaining = list(range(n_done, n_stars))

    print(f"\n{'='*65}")
    print(f"  Tier 3: NUTS with JIT-compiled log-prob")
    print(f"  {n_stars} stars, batch {BATCH_SIZE_STARS}")
    print(f"  Stage 1 (core {N_CORE}D): {NUM_CHAINS_CORE}ch x {NUM_RESULTS_CORE} samples, burn-in {NUM_BURNIN_CORE}, NUTS depth {MAX_TREE_DEPTH}")
    print(f"  Stage 2 ({N_ABUND} elem x 1D): {NUM_CHAINS_ELEM}ch x {NUM_RESULTS_ELEM} samples, burn-in {NUM_BURNIN_ELEM}")
    print(f"  To do: {len(remaining)} stars")
    print(f"{'='*65}\n")

    total_start = time.time()
    batch_count = 0

    for bstart in range(0, len(remaining), BATCH_SIZE_STARS):
        bloc = remaining[bstart:bstart + BATCH_SIZE_STARS]
        actual = len(bloc)
        bf = flux_array[bloc].astype(np.float32)
        bi = ivar_array[bloc].astype(np.float32)
        cnn_mu, cnn_sig = cnn_predict_physical(bf)

        if actual < BATCH_SIZE_STARS:
            pad = BATCH_SIZE_STARS - actual
            bf = np.concatenate([bf, np.zeros((pad, N_PIXELS), np.float32)])
            bi = np.concatenate([bi, np.zeros((pad, N_PIXELS), np.float32)])
            cnn_mu  = np.concatenate([cnn_mu, np.tile(cnn_mu[:1], (pad, 1))])
            cnn_sig = np.concatenate([cnn_sig, np.tile(cnn_sig[:1], (pad, 1))])

        t_s1 = time.time()
        core_med, core_lo, core_hi, s1_acc, s1_rhat = run_core_mcmc(bf, bi, cnn_mu, cnn_mu, cnn_sig)
        t_s1e = time.time() - t_s1
        print(f"    S1 core: acc={s1_acc:.2f}  R-hat max={s1_rhat.max():.3f}  {t_s1e:.1f}s")

        fixed_23 = cnn_mu.copy()
        for ci, gi in enumerate(CORE_INDICES): fixed_23[:, gi] = core_med[:, ci]

        t_s2 = time.time()
        amed = np.zeros((BATCH_SIZE_STARS, N_ABUND), np.float32)
        alo = np.zeros_like(amed); ahi = np.zeros_like(amed)
        s2r, s2a = [], []
        for ai, ei in enumerate(ABUND_INDICES):
            em, el, eh, ea, er = run_element_mcmc(bf, bi, fixed_23, ei, cnn_mu, cnn_sig)
            amed[:, ai] = em; alo[:, ai] = el; ahi[:, ai] = eh
            s2r.append(er); s2a.append(ea)
            fixed_23[:, ei] = em
        t_s2e = time.time() - t_s2
        print(f"    S2 elem: acc={np.mean(s2a):.2f}  R-hat max={max(s2r):.3f}  {t_s2e:.1f}s")

        full_med = np.zeros((BATCH_SIZE_STARS, N_LABELS_RAW), np.float32)
        full_lo = np.zeros_like(full_med); full_hi = np.zeros_like(full_med)
        full_rhat = np.zeros(N_LABELS_RAW, np.float32)
        for ci, gi in enumerate(CORE_INDICES):
            full_med[:, gi] = core_med[:, ci]; full_lo[:, gi] = core_lo[:, ci]
            full_hi[:, gi] = core_hi[:, ci]; full_rhat[gi] = s1_rhat[ci]
        for ai, gi in enumerate(ABUND_INDICES):
            full_med[:, gi] = amed[:, ai]; full_lo[:, gi] = alo[:, ai]
            full_hi[:, gi] = ahi[:, ai]; full_rhat[gi] = s2r[ai]

        elapsed = t_s1e + t_s2e
        avg_acc = (s1_acc + np.mean(s2a)) / 2
        for b in range(actual):
            li = bloc[b]; gi = int(test_indices[li])
            results['global_indices'].append(gi)
            results['true_labels'].append(true_phys[li])
            results['inferred_labels'].append(full_med[b])
            results['lower_sigma'].append(full_lo[b])
            results['upper_sigma'].append(full_hi[b])
            results['aspcap_errors'].append(aspcap_errors[li, :N_LABELS_RAW])
            results['acceptance_rates'].append(avg_acc)
            results['r_hat'].append(full_rhat)
            results['wall_seconds'].append(elapsed / actual)

        n_done += actual; batch_count += 1
        save_checkpoint(results)
        el = (n_stars - n_done) * elapsed / actual / 60
        print(f"  [{n_done:>4}/{n_stars}]  {elapsed:.1f}s  (~{el:.0f} min left)\n")

    print(f"\nAll {n_stars} stars done in {(time.time()-total_start)/60:.1f} min.")
    save_final_results(results)
    return results


# ============================================================================
#  ALTERNATIVE: JOINT 23-PARAMETER NUTS PIPELINE
# ============================================================================

# 23D bijector
_joint_bij = tfb.Chain([tfb.Shift(bounds_low), tfb.Scale(bounds_high - bounds_low), tfb.Sigmoid()])
_joint_inv = tfb.Invert(_joint_bij)

_joint_cnn_mu_var  = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False)
_joint_cnn_std_var = tf.Variable(tf.ones([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False)
_joint_obs_flux_var = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_PIXELS], tf.float32), trainable=False)
_joint_obs_ivar_var = tf.Variable(tf.zeros([BATCH_SIZE_STARS, N_PIXELS], tf.float32), trainable=False)

NUM_CHAINS_JOINT = 6; NUM_RESULTS_JOINT = 400; NUM_BURNIN_JOINT = 400; NUM_ADAPT_JOINT = 200

_init_joint_var = tf.Variable(
    tf.zeros([NUM_CHAINS_JOINT, BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False)

@tf.function(reduce_retracing=True)
def joint_log_prob_fn(theta_joint, obs_flux, obs_ivar):
    """theta_joint: (C, B, 23) physical -> (C, B)."""
    num_chains  = tf.shape(theta_joint)[0]
    batch_stars = tf.shape(theta_joint)[1]
    theta_BCL  = tf.transpose(theta_joint, [1, 0, 2])
    theta_flat = tf.reshape(theta_BCL, [-1, N_LABELS_RAW])
    labels_27   = get_27_features(theta_flat)
    labels_norm = (labels_27 - MEAN_TENSOR) / STD_TENSOR
    model_flux = _forward_model_jit(labels_norm)
    obs_flux_rep = tf.repeat(obs_flux, num_chains, axis=0)
    obs_ivar_rep = tf.repeat(obs_ivar, num_chains, axis=0)
    safe_flux = tf.where(tf.math.is_finite(obs_flux_rep), obs_flux_rep, 0.0)
    safe_ivar = tf.where(
        tf.math.is_finite(obs_ivar_rep) & tf.math.is_finite(obs_flux_rep),
        obs_ivar_rep, 0.0)
    safe_ivar = 1.0 / (1.0 / (safe_ivar + 1e-12) + 1e-5)
    fv = tf.cast((safe_flux > 1e-3) & (safe_flux < 1.3), tf.float32)
    iv = tf.cast(safe_ivar > 0.0, tf.float32)
    mask = fv * iv
    chi2 = tf.reduce_sum(tf.square(safe_flux - model_flux) * safe_ivar * mask, axis=1)
    log_lik = -0.5 * chi2
    cnn_mu_rep  = tf.repeat(_joint_cnn_mu_var, num_chains, axis=0)
    cnn_std_rep = tf.repeat(_joint_cnn_std_var, num_chains, axis=0)
    log_prior = -0.5 * tf.reduce_sum(tf.square((theta_flat - cnn_mu_rep) / cnn_std_rep), axis=1)
    log_post = log_lik + PRIOR_WEIGHT * log_prior
    return tf.transpose(tf.reshape(log_post, [batch_stars, num_chains]))

#@tf.function(reduce_retracing=True)
def sample_joint_compiled():
    """NUTS for joint 23D inference."""
    def log_prob_closure(theta_unc):
        return joint_log_prob_fn(
            _joint_bij.forward(theta_unc), _joint_obs_flux_var, _joint_obs_ivar_var)
    step_size = tf.fill([1, BATCH_SIZE_STARS, N_LABELS_RAW], 0.05)
    nuts = tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=log_prob_closure,
        step_size=step_size,
        max_tree_depth=MAX_TREE_DEPTH,
    )
    adaptive = tfp.mcmc.DualAveragingStepSizeAdaptation(
        inner_kernel=nuts,
        num_adaptation_steps=NUM_ADAPT_JOINT,
        target_accept_prob=TARGET_ACCEPT,
        step_size_setter_fn=lambda pkr, s: pkr._replace(step_size=s),
        step_size_getter_fn=lambda pkr: pkr.step_size,
        log_accept_prob_getter_fn=lambda pkr: pkr.log_accept_ratio,
    )
    init_unc = _joint_inv.forward(_init_joint_var)
    return tfp.mcmc.sample_chain(
        num_results=NUM_RESULTS_JOINT,
        num_burnin_steps=NUM_BURNIN_JOINT,
        current_state=init_unc,
        kernel=adaptive,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted,
    )

def run_inference_pipeline_alt(test_indices, true_labels_norm, aspcap_errors,
                               flux_array, ivar_array):
    CHECKPOINT_ALT_PATH = os.path.join(RESULTS_DIR, "checkpoint.npz")
    RESULTS_ALT_PATH    = os.path.join(RESULTS_DIR, "results.npz")

    n_stars = len(test_indices)
    true_phys = true_labels_norm[:, :N_LABELS_RAW] * STD_TENSOR[:N_LABELS_RAW] + MEAN_TENSOR[:N_LABELS_RAW]

    fresh = {'global_indices':[],'true_labels':[],'inferred_labels':[],
             'lower_sigma':[],'upper_sigma':[],'aspcap_errors':[],
             'acceptance_rates':[],'r_hat':[],'wall_seconds':[]}
    if os.path.exists(CHECKPOINT_ALT_PATH):
        data = np.load(CHECKPOINT_ALT_PATH, allow_pickle=True)
        results = {k: list(data[k]) for k in fresh}
        n_done = len(results['global_indices'])
        print(f"  Resuming (Joint NUTS): {n_done} done.")
    else:
        results = fresh; n_done = 0
    remaining = list(range(n_done, n_stars))

    print(f"\n{'='*65}")
    print(f"  Tier 3-Alt: JOINT NUTS 23D with JIT-compiled log-prob")
    print(f"  {n_stars} stars, batch {BATCH_SIZE_STARS}")
    print(f"  Joint: {NUM_CHAINS_JOINT}ch x {NUM_RESULTS_JOINT} samples, burn-in {NUM_BURNIN_JOINT}, depth {MAX_TREE_DEPTH}")
    print(f"  To do: {len(remaining)} stars")
    print(f"{'='*65}\n")

    total_start = time.time()

    for bstart in range(0, len(remaining), BATCH_SIZE_STARS):
        bloc = remaining[bstart:bstart + BATCH_SIZE_STARS]
        actual = len(bloc)
        bf = flux_array[bloc].astype(np.float32)
        bi = ivar_array[bloc].astype(np.float32)
        cnn_mu, cnn_sig = cnn_predict_physical(bf)

        if actual < BATCH_SIZE_STARS:
            pad = BATCH_SIZE_STARS - actual
            bf = np.concatenate([bf, np.zeros((pad, N_PIXELS), np.float32)])
            bi = np.concatenate([bi, np.zeros((pad, N_PIXELS), np.float32)])
            cnn_mu  = np.concatenate([cnn_mu, np.tile(cnn_mu[:1], (pad, 1))])
            cnn_sig = np.concatenate([cnn_sig, np.tile(cnn_sig[:1], (pad, 1))])

        _joint_cnn_mu_var.assign(tf.cast(cnn_mu, tf.float32))
        _joint_cnn_std_var.assign(tf.cast(cnn_sig, tf.float32))
        _joint_obs_flux_var.assign(tf.cast(bf, tf.float32))
        _joint_obs_ivar_var.assign(tf.cast(bi, tf.float32))

        margin = 1e-3
        joint_init_phys = tf.clip_by_value(
            tf.cast(cnn_mu, tf.float32), bounds_low + margin, bounds_high - margin)
        init_state = tf.tile(tf.expand_dims(joint_init_phys, 0), [NUM_CHAINS_JOINT, 1, 1])
        joint_std = STD_TENSOR[:N_LABELS_RAW]
        init_state = init_state + tf.random.normal(tf.shape(init_state)) * (0.05 * joint_std)
        init_state = tf.clip_by_value(init_state, bounds_low + margin, bounds_high - margin)
        _init_joint_var.assign(init_state)

        t0 = time.time()
        samples_unc, is_accepted = sample_joint_compiled()
        samples_phys = _joint_bij.forward(samples_unc)
        r_hat_all = tf.reduce_mean(
            tfp.mcmc.diagnostic.potential_scale_reduction(samples_phys, independent_chain_ndims=1), axis=0
        ).numpy()
        flat = tf.reshape(samples_phys, [NUM_RESULTS_JOINT * NUM_CHAINS_JOINT, -1, N_LABELS_RAW])
        pcts = tfp.stats.percentile(flat, [16., 50., 84.], axis=0)
        joint_med = pcts[1].numpy()
        joint_lo = (pcts[1]-pcts[0]).numpy()
        joint_hi = (pcts[2]-pcts[1]).numpy()
        avg_acc = tf.reduce_mean(tf.cast(is_accepted, tf.float32)).numpy()
        elapsed = time.time() - t0
        print(f"    Joint NUTS ({NUM_RESULTS_JOINT} samples): {elapsed:.2f}s  acc={avg_acc:.2f}  R-hat max={r_hat_all.max():.3f}")

        for b in range(actual):
            li = bloc[b]; gi = int(test_indices[li])
            results['global_indices'].append(gi)
            results['true_labels'].append(true_phys[li])
            results['inferred_labels'].append(joint_med[b])
            results['lower_sigma'].append(joint_lo[b])
            results['upper_sigma'].append(joint_hi[b])
            results['aspcap_errors'].append(aspcap_errors[li, :N_LABELS_RAW])
            results['acceptance_rates'].append(avg_acc)
            results['r_hat'].append(r_hat_all)
            results['wall_seconds'].append(elapsed / actual)

        n_done += actual
        np.savez(CHECKPOINT_ALT_PATH, **{k: np.array(v) for k, v in results.items()})
        el = (n_stars - n_done) * elapsed / actual / 60
        print(f"  [{n_done:>4}/{n_stars}]  {elapsed:.1f}s  (~{el:.0f} min left)\n")

    print(f"\nAll {n_stars} stars done in {(time.time()-total_start)/60:.1f} min.")
    arrays = {k: np.array(v) for k, v in results.items()}
    arrays['label_names'] = np.array(LABEL_NAMES)
    np.savez(RESULTS_ALT_PATH, **arrays)
    print(f"Final results saved to {RESULTS_ALT_PATH}")
    return results


In [3]:
if __name__ == "__main__":

    def select_stratified_test_sample(h5_path, stats_path, selected_labels,
                                       test_start, test_end, target_n=5,
                                       logg_bins=5, teff_bins=10, mh_bins=10):
        """Selects stratified test stars."""
        max_bins = logg_bins * teff_bins * mh_bins
        with h5py.File(h5_path, 'r') as f:
            meta = f['metadata']
            teff = meta['TEFF'][test_start:test_end].astype(np.float32)
            logg = meta['LOGG'][test_start:test_end].astype(np.float32)
            m_h  = meta['M_H'][test_start:test_end].astype(np.float32)
            snr  = meta['SNR'][test_start:test_end].astype(np.float32)

        SENTINEL = -5000.0
        valid = (teff > SENTINEL) & (logg > SENTINEL) & (m_h > SENTINEL)

        def percentile_edges(values, mask, n_bins):
            pts = np.linspace(0, 100, n_bins + 1)
            edges = np.percentile(values[mask], pts)
            edges[0] -= 1e-6; edges[-1] += 1e-6
            return edges

        logg_edges = percentile_edges(logg, valid, logg_bins)
        teff_edges = percentile_edges(teff, valid, teff_bins)
        mh_edges   = percentile_edges(m_h, valid, mh_bins)

        logg_bin = np.clip(np.digitize(logg, logg_edges) - 1, 0, logg_bins - 1)
        teff_bin = np.clip(np.digitize(teff, teff_edges) - 1, 0, teff_bins - 1)
        mh_bin   = np.clip(np.digitize(m_h, mh_edges) - 1, 0, mh_bins - 1)

        local_indices_selected = []
        bin_summary = []
        for i in range(logg_bins):
            for j in range(teff_bins):
                for k in range(mh_bins):
                    in_bin = valid & (logg_bin == i) & (teff_bin == j) & (mh_bin == k)
                    idx = np.where(in_bin)[0]
                    if len(idx) == 0: continue
                    median_snr = np.median(snr[idx])
                    closest = idx[np.argmin(np.abs(snr[idx] - median_snr))]
                    local_indices_selected.append(closest)
                    bin_summary.append({'n_stars': len(idx)})

        local_indices_selected = np.array(local_indices_selected)
        n_from_bins = len(local_indices_selected)

        if n_from_bins < target_n:
            shortfall = target_n - n_from_bins
            remaining = np.array([i for i in np.where(valid)[0]
                                  if i not in set(local_indices_selected.tolist())])
            rng = np.random.default_rng(42)
            extra = rng.choice(remaining, size=min(shortfall, len(remaining)), replace=False)
            local_indices_selected = np.concatenate([local_indices_selected, extra])
        elif n_from_bins > target_n:
            occupancies = np.array([b['n_stars'] for b in bin_summary])
            order = np.argsort(occupancies)
            local_indices_selected = local_indices_selected[order[:target_n]]

        global_indices = np.sort(local_indices_selected + test_start)
        print(f"Selected {len(global_indices)} test stars.")
        return global_indices

    TEST_START = 140_000
    TEST_END   = 149_500

    global_indices = select_stratified_test_sample(
        config.H5_PATH, config.STATS_PATH, config.SELECTED_LABELS,
        TEST_START, TEST_END, target_n=TOTAL_STARS, logg_bins=5, teff_bins=10, mh_bins=10,
    )
    np.save("test_indices.npy", global_indices)

    test_indices = np.load("test_indices.npy")
    results = run_inference_pipeline_alt(
        test_indices=test_indices,
        true_labels_norm=labels,
        aspcap_errors=label_err,
        flux_array=real_data,
        ivar_array=real_data_ivar,
    )


Selected 10 test stars.

  Tier 3-Alt: JOINT NUTS 23D with JIT-compiled log-prob
  10 stars, batch 10
  Joint: 6ch x 400 samples, burn-in 400, depth 10
  To do: 10 stars



I0000 00:00:1776353024.728851      23 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776353028.490668      68 service.cc:152] XLA service 0x799e888fe650 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776353028.490700      68 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776353028.798200      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    Joint NUTS (400 samples): 8641.89s  acc=1.00  R-hat max=8.126
  [  10/10]  8641.9s  (~0 min left)


All 10 stars done in 144.1 min.
Final results saved to /kaggle/working/nuts_results/results.npz


In [4]:
"""
nuts_results_analysis.py — Load, report and visualise NUTS inference results
==========================================================================
Designed to work with results.npz produced by the NUTS inference pipeline.
"""
import os
import textwrap
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import LogNorm

# CONSTANTS
PRETTY_NAMES = {
    "TEFF": r"$T_{\rm eff}$ [K]", "LOGG": r"$\log g$", "M_H": "[M/H]",
    "VMICRO": r"$v_{\rm micro}$ [km/s]", "VMACRO": r"$v_{\rm macro}$ [km/s]",
    "VSINI": r"$v \sin i$ [km/s]", "C_FE": "[C/Fe]", "N_FE": "[N/Fe]",
    "O_FE": "[O/Fe]", "FE_H": "[Fe/H]", "MG_FE": "[Mg/Fe]", "SI_FE": "[Si/Fe]",
    "CA_FE": "[Ca/Fe]", "TI_FE": "[Ti/Fe]", "S_FE": "[S/Fe]", "AL_FE": "[Al/Fe]",
    "MN_FE": "[Mn/Fe]", "NI_FE": "[Ni/Fe]", "CR_FE": "[Cr/Fe]", "K_FE": "[K/Fe]",
    "NA_FE": "[Na/Fe]", "V_FE": "[V/Fe]", "CO_FE": "[Co/Fe]",
}
CORE_LABELS  = ["TEFF", "LOGG", "M_H", "VMICRO", "VMACRO", "VSINI", "C_FE", "N_FE", "O_FE"]
ABUND_LABELS = ["FE_H", "MG_FE", "SI_FE", "CA_FE", "TI_FE", "S_FE",
                "AL_FE", "MN_FE", "NI_FE", "CR_FE", "K_FE", "NA_FE", "V_FE", "CO_FE"]

_C_CORE = "#2c7bb6"; _C_ABUND = "#d7191c"; _C_HIST = "#636363"; _C_UNITY = "#1a1a1a"

def _nice_label(name): return PRETTY_NAMES.get(name, name)
def _iqr(x): return np.percentile(x, 75) - np.percentile(x, 25)
def _save(fig, save_dir, stem):
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        for ext in ("png", "pdf"):
            fig.savefig(os.path.join(save_dir, f"{stem}.{ext}"), dpi=200, bbox_inches="tight")

def load_map_results(path):
    raw = np.load(path, allow_pickle=True)
    label_names = [s.item() if isinstance(s, np.generic) else str(s) for s in raw["label_names"]]
    true = raw["true_labels"].astype(np.float64)
    inferred = raw["inferred_labels"].astype(np.float64)
    aspcap_err = raw["aspcap_errors"].astype(np.float64)
    wall = raw["wall_seconds"].astype(np.float64)
    gidx = raw["global_indices"]
    # NUTS-specific: load posterior widths
    lo_sig = raw["lower_sigma"].astype(np.float64) if "lower_sigma" in raw.files else None
    hi_sig = raw["upper_sigma"].astype(np.float64) if "upper_sigma" in raw.files else None
    r_hat = raw["r_hat"].astype(np.float64) if "r_hat" in raw.files else None
    acc = raw["acceptance_rates"].astype(np.float64) if "acceptance_rates" in raw.files else None
    return {
        "label_names": label_names, "n_stars": len(gidx), "n_labels": len(label_names),
        "true_labels": true, "inferred_labels": inferred, "residuals": inferred - true,
        "aspcap_errors": aspcap_err, "wall_seconds": wall, "global_indices": gidx,
        "lower_sigma": lo_sig, "upper_sigma": hi_sig, "r_hat": r_hat, "acceptance_rates": acc,
    }

def report_map_results(data, print_report=True):
    label_names = data["label_names"]; residuals = data["residuals"]
    aspcap_err = data["aspcap_errors"]; n_stars = data["n_stars"]; wall = data["wall_seconds"]
    report = {}
    for j, name in enumerate(label_names):
        r = residuals[:, j]
        report[name] = {
            "bias": np.median(r), "mad": np.median(np.abs(r - np.median(r))),
            "rmse": np.sqrt(np.mean(r**2)), "iqr": _iqr(r),
            "median_aspcap_err": np.median(aspcap_err[:, j]), "n_stars": n_stars,
        }
    if print_report:
        hdr = f"{'Label':<10} {'Bias':>10} {'MAD':>10} {'RMSE':>10} {'IQR':>10} {'ASPCAP s':>10}"
        rule = "-" * len(hdr)
        lines = ["", "=" * 66, "  NUTS Results - Summary Statistics",
                 f"  {n_stars} stars,  median wall-time = {np.median(wall):.2f} s/star",
                 "=" * 66, "", "  Core physics parameters", f"  {rule}", f"  {hdr}", f"  {rule}"]
        for name in CORE_LABELS:
            if name not in report: continue
            s = report[name]
            lines.append(f"  {PRETTY_NAMES.get(name, name):<10} {s['bias']:>+10.4f} {s['mad']:>10.4f} {s['rmse']:>10.4f} {s['iqr']:>10.4f} {s['median_aspcap_err']:>10.4f}")
        lines += [f"  {rule}", "", "  Individual abundances", f"  {rule}", f"  {hdr}", f"  {rule}"]
        for name in ABUND_LABELS:
            if name not in report: continue
            s = report[name]
            lines.append(f"  {PRETTY_NAMES.get(name, name):<10} {s['bias']:>+10.4f} {s['mad']:>10.4f} {s['rmse']:>10.4f} {s['iqr']:>10.4f} {s['median_aspcap_err']:>10.4f}")
        lines.append(f"  {rule}")
        if data.get("acceptance_rates") is not None:
            lines.append(f"\n  NUTS acceptance rate: {np.mean(data['acceptance_rates']):.3f}")
        if data.get("r_hat") is not None:
            rh = np.array(data["r_hat"])
            if rh.ndim > 1: rh = rh.max(axis=0)
            lines.append(f"  R-hat max: {rh.max():.3f}")
        print("\n".join(lines))
    return report

def plot_one_to_one(data, save_dir=None):
    label_names = data["label_names"]; true = data["true_labels"]; inferred = data["inferred_labels"]
    n_labels = len(label_names); n_cols = 6; n_rows = int(np.ceil(n_labels / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3.2, n_rows*3.2))
    fig.patch.set_facecolor("white"); axes = axes.ravel()
    for j in range(n_labels):
        ax = axes[j]; t = true[:, j]; inf = inferred[:, j]; name = label_names[j]
        c = _C_CORE if name in CORE_LABELS else _C_ABUND
        ax.scatter(t, inf, s=12, alpha=0.65, edgecolors="none", c=c, zorder=2)
        lo = min(t.min(), inf.min()); hi = max(t.max(), inf.max())
        margin = 0.05*(hi-lo) if hi > lo else 0.5
        ax.plot([lo-margin, hi+margin], [lo-margin, hi+margin], ls="--", lw=1, c=_C_UNITY, alpha=0.5, zorder=1)
        ax.set_xlim(lo-margin, hi+margin); ax.set_ylim(lo-margin, hi+margin)
        ax.set_xlabel("ASPCAP", fontsize=8); ax.set_ylabel("NUTS", fontsize=8)
        ax.set_title(_nice_label(name), fontsize=9, fontweight="bold"); ax.tick_params(labelsize=7)
        ax.set_aspect("equal", adjustable="box")
    for j in range(n_labels, len(axes)): axes[j].set_visible(False)
    fig.suptitle("NUTS Inference - Inferred vs ASPCAP (True)", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout(); _save(fig, save_dir, "nuts_one_to_one"); return fig

def plot_residual_histograms(data, save_dir=None):
    label_names = data["label_names"]; residuals = data["residuals"]
    n_labels = len(label_names); n_cols = 6; n_rows = int(np.ceil(n_labels / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3.0, n_rows*2.8))
    fig.patch.set_facecolor("white"); axes = axes.ravel()
    for j in range(n_labels):
        ax = axes[j]; r = residuals[:, j]; name = label_names[j]
        c = _C_CORE if name in CORE_LABELS else _C_ABUND
        n_bins = max(10, min(50, int(np.sqrt(len(r)))))
        ax.hist(r, bins=n_bins, color=c, alpha=0.75, edgecolor="white", linewidth=0.5)
        ax.axvline(0, ls="--", lw=1.0, c=_C_UNITY, alpha=0.5)
        ax.axvline(np.median(r), ls="-", lw=1.2, c="k", alpha=0.8, label=f"med={np.median(r):.3f}")
        ax.set_xlabel("Residual", fontsize=8); ax.set_title(_nice_label(name), fontsize=9, fontweight="bold")
        ax.tick_params(labelsize=7); ax.legend(fontsize=6, loc="upper right")
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    for j in range(n_labels, len(axes)): axes[j].set_visible(False)
    fig.suptitle("NUTS Inference - Residual Distributions", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout(); _save(fig, save_dir, "nuts_residual_histograms"); return fig

def plot_bias_scatter_bars(data, save_dir=None):
    label_names = data["label_names"]; residuals = data["residuals"]; n_labels = len(label_names)
    biases = np.array([np.median(residuals[:, j]) for j in range(n_labels)])
    mads = np.array([np.median(np.abs(residuals[:, j] - biases[j])) for j in range(n_labels)])
    order = np.arange(n_labels)[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.45*n_labels+1.5)); fig.patch.set_facecolor("white")
    colours = [_C_CORE if label_names[j] in CORE_LABELS else _C_ABUND for j in order]
    ax.barh(range(n_labels), biases[order], xerr=mads[order], color=colours, alpha=0.8,
            edgecolor="white", linewidth=0.5, error_kw=dict(lw=1, capsize=3, capthick=1, ecolor="#333"))
    ax.axvline(0, ls="-", lw=0.8, c=_C_UNITY, alpha=0.5); ax.set_yticks(range(n_labels))
    ax.set_yticklabels([_nice_label(label_names[j]) for j in order], fontsize=8)
    ax.set_xlabel("Median bias +/- MAD", fontsize=10)
    ax.set_title("NUTS Inference - Per-Label Bias & Scatter", fontsize=13, fontweight="bold")
    fig.tight_layout(); _save(fig, save_dir, "nuts_bias_scatter_bars"); return fig

def plot_residual_vs_teff(data, save_dir=None):
    label_names = data["label_names"]; true = data["true_labels"]; residuals = data["residuals"]
    teff = true[:, label_names.index("TEFF")]
    abund_indices = [j for j, n in enumerate(label_names) if n in ABUND_LABELS]
    n = len(abund_indices); n_cols = 5; n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3.2, n_rows*2.8), sharey=False)
    fig.patch.set_facecolor("white"); axes = axes.ravel()
    for k, j in enumerate(abund_indices):
        ax = axes[k]; r = residuals[:, j]
        ax.scatter(teff, r, c=teff, cmap="RdYlBu_r", s=10, alpha=0.7, edgecolors="none")
        ax.axhline(0, ls="--", lw=0.8, c=_C_UNITY, alpha=0.4)
        ax.set_xlabel(r"$T_{\rm eff}$ [K]", fontsize=7); ax.set_ylabel("Residual", fontsize=7)
        ax.set_title(_nice_label(label_names[j]), fontsize=9, fontweight="bold"); ax.tick_params(labelsize=6)
    for k in range(n, len(axes)): axes[k].set_visible(False)
    fig.suptitle("NUTS Residuals vs Teff", fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout(); _save(fig, save_dir, "nuts_residual_vs_teff"); return fig

def plot_residual_heatmap(data, save_dir=None):
    label_names = data["label_names"]; residuals = data["residuals"]; n_stars = data["n_stars"]
    mads = np.array([np.median(np.abs(residuals[:, j] - np.median(residuals[:, j]))) for j in range(len(label_names))])
    mads = np.where(mads < 1e-8, 1.0, mads); normed = np.abs(residuals) / mads[None, :]
    fig, ax = plt.subplots(figsize=(max(6, 0.15*n_stars+2), 7)); fig.patch.set_facecolor("white")
    im = ax.imshow(normed.T, aspect="auto", cmap="inferno", interpolation="nearest", vmin=0, vmax=5)
    ax.set_yticks(range(len(label_names))); ax.set_yticklabels([_nice_label(n) for n in label_names], fontsize=7)
    ax.set_xlabel("Star index", fontsize=10); ax.set_title("NUTS |Residuals| / MAD (per label)", fontsize=13, fontweight="bold")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02); cbar.set_label("x MAD", fontsize=9)
    fig.tight_layout(); _save(fig, save_dir, "nuts_residual_heatmap"); return fig

def plot_wall_time(data, save_dir=None):
    wall = data["wall_seconds"]
    fig, ax = plt.subplots(figsize=(6, 3.5)); fig.patch.set_facecolor("white")
    ax.hist(wall, bins=max(10, min(40, len(wall)//3)), color="#636363", alpha=0.8, edgecolor="white", linewidth=0.5)
    ax.axvline(np.median(wall), ls="-", lw=1.5, c=_C_CORE, label=f"median = {np.median(wall):.2f} s")
    ax.set_xlabel("Wall time per star [s]", fontsize=10); ax.set_ylabel("Count", fontsize=10)
    ax.set_title("NUTS Inference - Per-Star Timing", fontsize=13, fontweight="bold"); ax.legend(fontsize=9)
    fig.tight_layout(); _save(fig, save_dir, "nuts_wall_time"); return fig

def plot_rmse_vs_aspcap(data, save_dir=None):
    label_names = data["label_names"]; residuals = data["residuals"]; aspcap_err = data["aspcap_errors"]
    n_labels = len(label_names)
    rmses = np.array([np.sqrt(np.mean(residuals[:, j]**2)) for j in range(n_labels)])
    med_aspcap = np.array([np.median(aspcap_err[:, j]) for j in range(n_labels)])
    x = np.arange(n_labels); w = 0.35
    fig, ax = plt.subplots(figsize=(14, 5)); fig.patch.set_facecolor("white")
    ax.bar(x-w/2, rmses, w, label="NUTS RMSE", color=_C_CORE, alpha=0.85, edgecolor="white")
    ax.bar(x+w/2, med_aspcap, w, label="ASPCAP median s", color=_C_ABUND, alpha=0.65, edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels([_nice_label(n) for n in label_names], fontsize=7, rotation=45, ha="right")
    ax.set_ylabel("Error scale", fontsize=10); ax.set_title("NUTS RMSE vs ASPCAP Pipeline Errors", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9); fig.tight_layout(); _save(fig, save_dir, "nuts_rmse_vs_aspcap"); return fig

def visualise_map_results(data, save_dir=None):
    figs = {}
    figs["one_to_one"] = plot_one_to_one(data, save_dir)
    figs["residual_histograms"] = plot_residual_histograms(data, save_dir)
    figs["bias_scatter_bars"] = plot_bias_scatter_bars(data, save_dir)
    figs["residual_vs_teff"] = plot_residual_vs_teff(data, save_dir)
    figs["residual_heatmap"] = plot_residual_heatmap(data, save_dir)
    figs["wall_time"] = plot_wall_time(data, save_dir)
    figs["rmse_vs_aspcap"] = plot_rmse_vs_aspcap(data, save_dir)
    if save_dir: print(f"\n  7 figures saved to {save_dir}/")
    return figs

def flag_outliers(data, sigma_thresh=3.0):
    label_names = data["label_names"]; residuals = data["residuals"]; n_stars = data["n_stars"]
    is_outlier = np.zeros(n_stars, dtype=bool); outlier_info = {}
    for j, name in enumerate(label_names):
        r = residuals[:, j]; med = np.median(r); mad = max(np.median(np.abs(r - med)), 1e-8)
        z = np.abs(r - med) / (1.4826 * mad); flagged = z > sigma_thresh
        is_outlier |= flagged
        outlier_info[name] = {"z_scores": z, "flagged": flagged, "n_flagged": int(flagged.sum())}
    print(f"\n  Outlier summary (>{sigma_thresh:.1f}s on any label):")
    print(f"  Total stars flagged: {is_outlier.sum()} / {n_stars} ({100*is_outlier.sum()/n_stars:.1f}%)")
    for name in CORE_LABELS + ABUND_LABELS:
        if name in outlier_info:
            print(f"  {PRETTY_NAMES.get(name,name):<10} {outlier_info[name]['n_flagged']:>10}")
    return is_outlier, outlier_info

def report_clean_vs_outlier(data, is_outlier):
    label_names = data["label_names"]; residuals = data["residuals"]
    clean_mask = ~is_outlier; n_clean = clean_mask.sum(); n_outlier = is_outlier.sum()
    print(f"\n  Clean: {n_clean} stars    Outlier: {n_outlier} stars")
    for j, name in enumerate(label_names):
        r_c = residuals[clean_mask, j]; r_o = residuals[is_outlier, j]
        rmse_c = np.sqrt(np.mean(r_c**2)) if len(r_c) > 0 else 0
        rmse_o = np.sqrt(np.mean(r_o**2)) if len(r_o) > 0 else 0
        ratio = rmse_o / rmse_c if rmse_c > 1e-10 else float('inf')
        print(f"  {PRETTY_NAMES.get(name, name):<10} {rmse_c:>12.4f} {rmse_o:>14.4f} {ratio:>7.1f}x")

def main(results_path, save_path):
    data = load_map_results(results_path)
    report = report_map_results(data)
    figs = visualise_map_results(data, save_dir=save_path)
    print(f"\nDone. {len(figs)} figures generated.")
    is_outlier, outlier_info = flag_outliers(data, sigma_thresh=3.0)
    report_clean_vs_outlier(data, is_outlier)

if __name__ == "__main__":
    main("/kaggle/working/nuts_results/results.npz", "visresults")



  NUTS Results - Summary Statistics
  10 stars,  median wall-time = 864.19 s/star

  Core physics parameters
  -----------------------------------------------------------------
  Label            Bias        MAD       RMSE        IQR   ASPCAP s
  -----------------------------------------------------------------
  $T_{\rm eff}$ [K]   -13.2587    11.1639    36.9037    30.0076     9.9926
  $\log g$      -0.0434     0.0359     0.1261     0.0821     0.0247
  [M/H]         -0.0121     0.0248     0.0272     0.0454     0.0074
  $v_{\rm micro}$ [km/s]    +0.0021     0.0150     0.0546     0.0689     0.0000
  $v_{\rm macro}$ [km/s]    +0.0216     0.0231     0.0614     0.0502     0.0000
  $v \sin i$ [km/s]    -1.5738     2.0318     2.5528     3.7854     0.0000
  [C/Fe]        -0.0016     0.0182     0.0629     0.0623     0.0144
  [N/Fe]        -0.0284     0.0389     0.1211     0.0887     0.0175
  [O/Fe]        -0.0005     0.0191     0.0283     0.0339     0.0193
  ----------------------------------

In [5]:
# ============================================================
# NUTS Uncertainty Diagnostics + Visualization
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.special import erf
    def _std_norm_cdf(x):
        x = np.asarray(x, dtype=np.float64)
        return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))
except ImportError:
    import math
    def _std_norm_cdf(x):
        x = np.asarray(x, dtype=np.float64)
        return 0.5 * (1.0 + np.vectorize(lambda t: math.erf(t / np.sqrt(2.0)))(x))

def _std_norm_pdf(x):
    return np.exp(-0.5 * x**2) / np.sqrt(2*np.pi)

def _stack(x):
    arr = np.asarray(x)
    if arr.dtype == object: arr = np.stack([np.asarray(v) for v in x])
    return arr.astype(np.float64)

def _scale(name): return 100.0 if str(name) == "TEFF" else 1.0

def load_results_nuts(results=None, path=None):
    if results is None:
        raw = np.load(path, allow_pickle=True)
        data = {k: raw[k] for k in raw.files}
    else:
        data = results
    # For NUTS: use (lower_sigma + upper_sigma) / 2 as symmetric error estimate
    lo = _stack(data["lower_sigma"])
    hi = _stack(data["upper_sigma"])
    err = (lo + hi) / 2.0
    return {
        "true": _stack(data["true_labels"]),
        "pred": _stack(data["inferred_labels"]),
        "err":  err,
        "names": list(data.get("label_names", globals().get("LABEL_NAMES")))
    }

def compute_metrics(d):
    true, pred, err, names = d["true"], d["pred"], d["err"], d["names"]
    out = {}
    for j, name in enumerate(names):
        s = _scale(name)
        t = true[:, j] / s; p = pred[:, j] / s; e = np.maximum(err[:, j] / s, 1e-12)
        z = (p - t) / e
        out[name] = dict(
            bias=np.mean(p - t), mae=np.mean(np.abs(p - t)), rmse=np.sqrt(np.mean((p - t)**2)),
            chi2=np.mean(z**2), z_mean=np.mean(z), z_std=np.std(z),
            cov1=np.mean(np.abs(z) < 1), cov2=np.mean(np.abs(z) < 2), cov3=np.mean(np.abs(z) < 3),
            median_sigma=np.median(e)
        )
    return out

def print_report(metrics):
    print("=== NUTS Uncertainty Report ===\n")
    for k, m in metrics.items():
        print(f"{str(k):<10} bias={m['bias']:+.4f}  RMSE={m['rmse']:.4f}  "
              f"s={m['median_sigma']:.4f}  chi2={m['chi2']:.3f}  z_std={m['z_std']:.3f}  "
              f"C1={m['cov1']:.3f} C2={m['cov2']:.3f}")

def plot_all(d):
    true, pred, err, names = d["true"], d["pred"], d["err"], d["names"]
    n = len(names)
    # z histogram
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    x = np.linspace(-5, 5, 200)
    for j in range(n):
        s = _scale(names[j]); z = ((pred[:, j] - true[:, j])/s) / (err[:, j]/s)
        axes[j].hist(z, bins=30, density=True, alpha=0.6)
        axes[j].plot(x, _std_norm_pdf(x), 'k--'); axes[j].set_title(names[j])
    plt.show()
    # pred vs true with error bars
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for j in range(n):
        s = _scale(names[j]); t = true[:, j]/s; p = pred[:, j]/s; e = err[:, j]/s
        axes[j].errorbar(t, p, yerr=e, fmt='o', alpha=0.4)
        axes[j].plot([t.min(), t.max()], [t.min(), t.max()], 'k--'); axes[j].set_title(names[j])
    plt.show()
    # calibration
    for j in range(n):
        s = _scale(names[j]); z = ((pred[:, j] - true[:, j])/s) / (err[:, j]/s)
        t = np.linspace(0.1, 3, 50); emp = [(np.abs(z) < x).mean() for x in t]
        th = _std_norm_cdf(t) - _std_norm_cdf(-t)
        plt.plot(t, emp, label="emp"); plt.plot(t, th, '--', label="ideal")
        plt.title(f"Calibration: {names[j]}"); plt.legend(); plt.show()

def run_uncertainty(results=None, path=None):
    d = load_results_nuts(results, path)
    m = compute_metrics(d); print_report(m); plot_all(d); return m

# RUN
run_uncertainty(results)
run_uncertainty(path=RESULTS_PATH)


=== NUTS Uncertainty Report ===

TEFF       bias=-0.2454  RMSE=0.3690  s=0.0458  chi2=45.778  z_std=4.611  C1=0.200 C2=0.400
LOGG       bias=-0.0010  RMSE=0.1261  s=0.0109  chi2=426.383  z_std=20.496  C1=0.100 C2=0.200
M_H        bias=-0.0108  RMSE=0.0272  s=0.0042  chi2=29.700  z_std=5.361  C1=0.000 C2=0.100
VMICRO     bias=+0.0286  RMSE=0.0546  s=0.0091  chi2=132.044  z_std=10.181  C1=0.500 C2=0.700
VMACRO     bias=+0.0394  RMSE=0.0614  s=0.0099  chi2=29.596  z_std=4.084  C1=0.300 C2=0.500
VSINI      bias=-1.4028  RMSE=2.5528  s=0.1101  chi2=585.234  z_std=19.641  C1=0.000 C2=0.100
C_FE       bias=+0.0286  RMSE=0.0629  s=0.0055  chi2=78.040  z_std=7.823  C1=0.200 C2=0.300
N_FE       bias=-0.0671  RMSE=0.1211  s=0.0072  chi2=288.088  z_std=14.357  C1=0.000 C2=0.200
O_FE       bias=-0.0062  RMSE=0.0283  s=0.0045  chi2=26.067  z_std=5.034  C1=0.200 C2=0.400
FE_H       bias=-0.0008  RMSE=0.0211  s=0.0036  chi2=39.605  z_std=6.251  C1=0.100 C2=0.200
MG_FE      bias=-0.0008  RMSE=0.0291  s

{np.str_('TEFF'): {'bias': np.float64(-0.24538256835937594),
  'mae': np.float64(0.2515222167968755),
  'rmse': np.float64(0.36903675644373274),
  'chi2': np.float64(45.77814469628696),
  'z_mean': np.float64(-4.951852231346388),
  'z_std': np.float64(4.610564409613726),
  'cov1': np.float64(0.2),
  'cov2': np.float64(0.4),
  'cov3': np.float64(0.5),
  'median_sigma': np.float64(0.045755615234375)},
 np.str_('LOGG'): {'bias': np.float64(-0.000992119312286377),
  'mae': np.float64(0.09096559286117553),
  'rmse': np.float64(0.12609003665628687),
  'chi2': np.float64(426.383434093567),
  'z_mean': np.float64(2.5102506018484703),
  'z_std': np.float64(20.495903883690673),
  'cov1': np.float64(0.1),
  'cov2': np.float64(0.2),
  'cov3': np.float64(0.2),
  'median_sigma': np.float64(0.010872602462768555)},
 np.str_('M_H'): {'bias': np.float64(-0.010818469524383544),
  'mae': np.float64(0.02374458611011505),
  'rmse': np.float64(0.027213154330231814),
  'chi2': np.float64(29.70036251411417),
 